In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from brisc.manuscript_analysis import viral_library as virlib
from brisc.manuscript_analysis import start_density_sim as start_sim
from brisc.manuscript_analysis import rabies_cell_counting as rv_count
from brisc.manuscript_analysis import starter_cell_counting as sc_count
from brisc.manuscript_analysis import overview_image
from brisc.manuscript_analysis.utils import despine
from brisc.manuscript_analysis.utils import get_path, get_output_folder
import numpy as np
from pathlib import Path
import tifffile as tf

import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm



In [ ]:
DATA_ROOT = None # Path(r"E:\temp\brisc_data\brisc")
PROJECT_DIV = "barcode_diversity_analysis"
PROJECT_RAB = "rabies_barcoding"
MOUSE = "BRAC11816.2e"
USE_DOWNSAMPLED = False
# DATA_ROOT = Path("Z:")

# optional, can be None of the path to arial.ttf:
arial_font_path = "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"


In [ ]:
# 1. Set to a parent folder for local work, or None for lab discovery
DATA_ROOT = None 
# 2. Get the uniform save path for figures (uses rabies_barcoding as an anchor)
save_path = get_output_folder(DATA_ROOT)
# 3. Get individual project paths using get_path
# In the lab (DATA_ROOT=None), these will use flexiznam to find each project.
# Locally, these will append the project name to your DATA_ROOT.
data_path_div = get_path(PROJECT_DIV, DATA_ROOT)
data_path_rab = get_path(PROJECT_RAB, DATA_ROOT)


if arial_font_path is not None:
    arial_prop = fm.FontProperties(fname=arial_font_path)
    plt.rcParams["font.family"] = arial_prop.get_name()
    plt.rcParams.update({"mathtext.default": "regular"})  # make math mode also Arial
    fm.fontManager.addfont(arial_font_path)
    matplotlib.rcParams["pdf.fonttype"] = 42  # for pdfs

    

In [ ]:
lib_path = data_path_div / "collapsed_barcodes/"
libraries_zhang = {
    "Clark, 2021": virlib.load_library_data(lib_path, "clark_rabies", 2, "bowtie"),
    "Saunders, 2022": virlib.load_library_data(
        lib_path, "saunders_pseudotyped", 2, "bowtie"
    ),
    "Zhang, 2024": virlib.load_library_data(
        lib_path, "Wickersham_Chen_CCS_EnvA", 2, "bowtie"
    ),
    # "Shin, 2024 N2c 1": virlib.load_library_data(
    #     data_path, "CVSN2c_finalvirallibrary1", 2, "bowtie" # lower diversity
    # ),
    "Shin, 2024 N2c": virlib.load_library_data(
        lib_path, "CVSN2c_finalvirallibrary2", 2, "bowtie"
    ),
    "Shin, 2024 SADB19": virlib.load_library_data(
        lib_path, "SADB19_finalvirallibrary1", 2, "bowtie"
    ),
    # "Shin, 2024 SADB19 2": virlib.load_library_data(
    #     data_path, "SADB19_finalvirallibrary2", 2, "bowtie" # lower diversity
    # ),
    "RV2": virlib.load_library_data(lib_path, "R2", 2, "bowtie"),
    "RV35": virlib.load_library_data(lib_path, "RV35", 2, "bowtie"),
}

In [ ]:
# Load Data
mcherry_file = (
    data_path_rab
    / MOUSE
    / "z_project_70_100_ch2_median2x2x2.tif"
)
background_file = (
    data_path_rab
    / MOUSE
    / "z_project_70_100_ch3_median2x2x2.tif"
)

mcherry = tf.imread(mcherry_file)
background = tf.imread(background_file)

injection_centers = {
    "BRYC64.2h": np.array([567, 144, 864]),
    "BRYC64.2i": np.array([673, 205, 890]),
    "BRAC11816.2e": np.array([828, 65, 699]),
}

voxel_distances_sorted, cell_distances_sorted = rv_count.rv_cortical_cell_distances(
    inj_center=injection_centers[MOUSE],
    project=PROJECT_RAB,
    mouse=MOUSE,
    data_root=DATA_ROOT,
)
IMAGE_FILE = data_path_rab / MOUSE  / 'BRAC11816.2e_slide3_slice20_MAX_projection_9slices_rotated90_median2x2x2.tif'
starter_img = tf.imread(IMAGE_FILE)
starter_img_metadata = None

In [ ]:
plot_barcoded = False

fontsize_dict = {"title": 8, "label": 7, "tick": 6, "legend": 6}
pad_dict = {"label": 1, "tick": 1, "legend": 5}
line_width = 0.9
line_alpha = 1

cm = 1 / 2.54
rasterized = True

In [ ]:
save_fig = True

In [ ]:
fig = plt.figure(figsize=(17.4 * cm, 7 * cm), dpi=300)
if True:
    # box to see fig dimension inline
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xticks([])
    ax.set_yticks([])

ax_coronal_local_rabies1 = fig.add_axes([0, 0.06, 0.45, 1])
ax_coronal_local_rabies2 = fig.add_axes([0.23, 0, 0.25, 0.4])

ax_starter_detection = fig.add_axes([0.43, 0.58, 0.37, 0.4])
ax_presynaptic_density = fig.add_axes([0.6, 0.18, 0.13, 0.32])
ax_starter_density = fig.add_axes([0.82, 0.18, 0.18, 0.6])

# 1) Plot the virus rescue scaling abundance histograms
if True:
    # Cell density vs PhP.eB dilution
    im = sc_count.plot_starter_dilution_densities(
        ax_starter_density,
        label_fontsize=fontsize_dict["label"],
        tick_fontsize=fontsize_dict["tick"],
        data_root=data_path_rab
    )
    ax_starter_density.set_yscale("log")


if True:

    kwargs = dict(
        z_proj_size=None,
        vlim_mcherry=(2000, 5000),
        vlim_background=(0, 5000),
        zoom_row_lims=(0, 3500),
        zoom_col_lims=(0, 4500),
        use_downsampled=False
    )

    rv_count.plot_rv_coronal_slice(
        injection_centers[MOUSE],
        (ax_coronal_local_rabies1, ax_coronal_local_rabies2),
        mcherry,
        background,
        **kwargs
    )

    # confocal example
    pixel_size_um = 0.3107403

    sc_count.plot_starter_confocal(ax_starter_detection, starter_img, None)
    overview_image.add_scalebar(
        ax_starter_detection,
        downsample_factor=1,
        pixel_size_um=pixel_size_um,
        length_um=20,
        bar_height_px=7,
        margin_px=10,
        location="bottom left"
    )

    im = rv_count.plot_rabies_density(
        inj_center=injection_centers[MOUSE],
        ax=ax_presynaptic_density,
        label_fontsize=fontsize_dict["label"],
        tick_fontsize=fontsize_dict["tick"],
        data_root=data_path_rab,
        linewidth=line_width,
        voxel_distances_sorted=voxel_distances_sorted,
        cell_distances_sorted=cell_distances_sorted,
    )

if save_fig:
    fname = save_path / "supplementary_figure1_preperstarter_and_dilution_with_iv.pdf"
    fig.savefig(fname)
    print(f"Figure saved as {fname}")

In [ ]:
# Print numbers
import pandas as pd
import glob

cellcount_folder = data_path_rab / MOUSE / 'confocal' / 'cell_counting'

csv_files = glob.glob(str(cellcount_folder / "*.csv"))
total_starters = 0
for f in csv_files:
    df = pd.read_csv(f)
    total_starters += len(df)
    
print(f"Total manual starter cell count: {total_starters}")

In [ ]:
import xml.etree.ElementTree as ET

xml_file = data_path_rab / MOUSE / 'cellfinder_results_010' / 'CellsCurated.xml'
tree = ET.parse(xml_file)
root = tree.getroot()
xml_pts = []
for marker in root.findall(".//Marker"):
    x = float(marker.find("MarkerX").text)
    y = float(marker.find("MarkerY").text)
    z = float(marker.find("MarkerZ").text)
    xml_pts.append([z, y, x])
n_pre = len(xml_pts)
scale = np.array([5,1,1])
xml_pts = np.vstack(xml_pts) * scale[None,:]
print(f"Total number of presynaptic cells: {n_pre}")
print(f'{n_pre/total_starters:.1f} presynaptic cells per starter')

In [ ]:
from scipy.ndimage import gaussian_filter
import matplotlib.patches as patches
import numpy as np

# 1. Define resolution for the density grid (bin size in pixels/units)
# Adjust this based on how "tight" you want the center to be
bin_size = 20 
sigma = 3    # Smoothing kernel size (in bins)

# 2. Create a 3D histogram of the points
# pts is [Z, Y, X]
min_coords = [2000, 0, 2000]
max_coords = [4000,2000,4000]

bins = [
    np.arange(min_coords[0], max_coords[0] + bin_size, bin_size),
    np.arange(min_coords[1], max_coords[1] + bin_size, bin_size),
    np.arange(min_coords[2], max_coords[2] + bin_size, bin_size)
]

hist, edges = np.histogramdd(pts, bins=bins)

# 3. Smooth the distribution to find the "blob" center
smoothed_hist = gaussian_filter(hist, sigma=sigma)

# 4. Find the coordinate of the maximum density
max_idx = np.unravel_index(np.argmax(smoothed_hist), smoothed_hist.shape)

# Convert bin index back to spatial coordinates (using bin centers)
center_zyx = np.array([edges[i][max_idx[i]] + bin_size/2 for i in range(3)])

print(f"Estimated Injection Center (Z, Y, X): {center_zyx}")
radius_px = 500 
# 1. Calculate 3D Euclidean distances from the center
distances = np.linalg.norm(pts - center_zyx, axis=1)
n_cells_in_sphere = np.sum(distances <= radius_px)
print(f"Number of cells within 0.5mm (500um) sphere: {n_cells_in_sphere}, {n_cells_in_sphere/total_starters:.1f} cells per starter cell")


# 5. Visualize the center on your previous plots
projections = [
    (2, 1, 'X', 'Y'), 
    (2, 0, 'X', 'Z'), 
    (1, 0, 'Y', 'Z')
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (dim_h, dim_v, label_h, label_v) in enumerate(projections):
    axes[i].scatter(pts[:, dim_h], pts[:, dim_v], alpha=0.3, s=2, color='gray')
    
    # Plot the calculated center
    axes[i].scatter(center_zyx[dim_h], center_zyx[dim_v], color='red', marker='x', s=100, label='Center')
    
    axes[i].set_xlabel(f'{label_h} (px)')
    axes[i].set_ylabel(f'{label_v} (px)')
    axes[i].set_title(f'{label_h}-{label_v} Projection')
    circle = patches.Circle((center_zyx[dim_h], center_zyx[dim_v]), radius_px, 
        color='blue', fill=False, linestyle='--', linewidth=1.5, label='0.5mm radius')
    axes[i].add_patch(circle)

    axes[i].set_aspect('equal', adjustable='datalim')
    axes[i].legend()

plt.tight_layout()

